In [1]:
import pandas as pd
import numpy as np

teams_df = pd.read_csv('../data/processed/teams.csv')
clinical_df = pd.read_csv('../data/processed/clinical_games.csv')
shots_df = pd.read_csv('../data/processed/shots_enriched.csv')

teams_df['date'] = pd.to_datetime(teams_df['date']).dt.normalize()
teams_df = teams_df.sort_values(['team_name', 'date']).reset_index(drop=True)

# matchday index per team per season — needed for the min-games floor on expanding features
teams_df['matchday'] = teams_df.groupby(['team_name', 'league', 'year']).cumcount() + 1

print(teams_df[['team_name', 'league', 'year', 'date', 'matchday']].head(10))

/var/folders/c3/hm5hq4r93ygcpfzfc4gv1mkh0000gn/T/ipykernel_11674/121244621.py:5: DtypeWarning: Columns (0: derby_name) have mixed types. Specify dtype option on import or set low_memory=False.
  clinical_df = pd.read_csv('../data/processed/clinical_games.csv')


  team_name   league  year       date  matchday
0  AC Milan  serie_a  2020 2020-09-21         1
1  AC Milan  serie_a  2020 2020-09-27         2
2  AC Milan  serie_a  2020 2020-10-04         3
3  AC Milan  serie_a  2020 2020-10-17         4
4  AC Milan  serie_a  2020 2020-10-26         5
5  AC Milan  serie_a  2020 2020-11-01         6
6  AC Milan  serie_a  2020 2020-11-08         7
7  AC Milan  serie_a  2020 2020-11-22         8
8  AC Milan  serie_a  2020 2020-11-29         9
9  AC Milan  serie_a  2020 2020-12-06        10


In [2]:
import ast

def parse_ppda(val):
    d = ast.literal_eval(val) if isinstance(val, str) else val
    return d['att'] / d['def'] if d['def'] != 0 else np.nan

teams_df['ppda_value'] = teams_df['ppda'].apply(parse_ppda)
teams_df['ppda_allowed_value'] = teams_df['ppda_allowed'].apply(parse_ppda)

# extraordinary PPDA rows (>50) — same exclusion as Notebook 04, but here we mask rather than drop
# since dropping rows would break the per-team match sequence used for rolling windows
teams_df.loc[teams_df['ppda_value'] > 50, 'ppda_value'] = np.nan
teams_df.loc[teams_df['ppda_allowed_value'] > 50, 'ppda_allowed_value'] = np.nan

print(teams_df['ppda_value'].isna().sum(), 'masked/missing ppda rows')

50 masked/missing ppda rows


In [3]:
def build_pointintime_features(df, group_cols=['team_name', 'league', 'year'],
                                 metrics=None, windows=(5,)):
    """
    For each metric, builds:
      - {metric}_roll{w}   : mean over trailing w matches, shifted so match N excludes itself
      - {metric}_expanding : season-to-date mean, shifted, NaN until min_games reached
    Sorting must already be done by date within each team before calling this.
    """
    df = df.copy()
    grouped = df.groupby(group_cols)

    for metric in metrics:
        for w in windows:
            df[f'{metric}_roll{w}'] = grouped[metric].transform(
                lambda s: s.shift(1).rolling(window=w, min_periods=w).mean()
            )
        df[f'{metric}_expanding'] = grouped[metric].transform(
            lambda s: s.shift(1).expanding(min_periods=3).mean()
        )

    return df

roll_metrics = ['ppda_value', 'ppda_allowed_value', 'scored', 'missed', 'xG', 'xGA', 'deep', 'deep_allowed']

teams_df = build_pointintime_features(teams_df, metrics=roll_metrics, windows=(5,))

# sanity check: first match of every team-season should be all-NaN on both roll5 and expanding
first_matches = teams_df[teams_df['matchday'] == 1]
print(first_matches[[f'{m}_roll5' for m in roll_metrics]].isna().all())

ppda_value_roll5            True
ppda_allowed_value_roll5    True
scored_roll5                True
missed_roll5                True
xG_roll5                    True
xGA_roll5                   True
deep_roll5                  True
deep_allowed_roll5          True
dtype: bool


In [4]:
matches_meta = pd.read_csv('../data/processed/matches.csv')
matches_meta['datetime'] = pd.to_datetime(matches_meta['datetime'])
matches_meta['date'] = matches_meta['datetime'].dt.normalize()

# long form: one row per (match_id, team_name, h_a) to merge onto teams_df
home_lookup = matches_meta[['match_id', 'date', 'league', 'year', 'home_team']].rename(columns={'home_team': 'team_name'})
home_lookup['h_a'] = 'h'
away_lookup = matches_meta[['match_id', 'date', 'league', 'year', 'away_team']].rename(columns={'away_team': 'team_name'})
away_lookup['h_a'] = 'a'
match_id_long = pd.concat([home_lookup, away_lookup], ignore_index=True)

teams_df = teams_df.merge(
    match_id_long,
    on=['date', 'league', 'year', 'team_name', 'h_a'],
    how='left'
)

print(teams_df['match_id'].isna().sum(), 'teams_df rows with no match_id match — should be 0 or very small')
print(teams_df.shape)

0 teams_df rows with no match_id match — should be 0 or very small
(17964, 43)


In [5]:
home_rows = teams_df[teams_df['h_a'] == 'h'].copy()
away_rows = teams_df[teams_df['h_a'] == 'a'].copy()

feature_cols = [f'{m}_roll5' for m in roll_metrics] + [f'{m}_expanding' for m in roll_metrics]

home_side = home_rows[['match_id', 'league', 'year', 'team_name', 'result'] + feature_cols].rename(
    columns={'team_name': 'home_team', 'result': 'home_result', **{c: f'home_{c}' for c in feature_cols}}
)
away_side = away_rows[['match_id', 'league', 'year', 'team_name', 'result'] + feature_cols].rename(
    columns={'team_name': 'away_team', 'result': 'away_result', **{c: f'away_{c}' for c in feature_cols}}
)

matches = home_side.merge(away_side, on=['match_id', 'league', 'year'], how='inner')

print(len(home_rows), len(away_rows), len(matches))  # should now be equal

8982 8982 8982


In [6]:
# 1. dtype mismatch (e.g. league strings differ in case/format between the two tables)
print(teams_df['league'].unique())
print(matches_meta['league'].unique())

# 2. date mismatch — check if datetime normalization actually aligns
print(teams_df['date'].head(3))
print(match_id_long['date'].head(3))
print(teams_df['date'].dtype, match_id_long['date'].dtype)

# 3. team_name spelling — pick one team and compare literal strings
print(sorted(teams_df['team_name'].unique())[:10])
print(sorted(match_id_long['team_name'].unique())[:10])

# 4. direct test on a single known match — does *anything* line up?
sample = teams_df[teams_df['h_a']=='h'].iloc[0]
print(sample[['team_name','date','league','year']])
print(match_id_long[(match_id_long['team_name']==sample['team_name']) & (match_id_long['h_a']=='h')].head())

<ArrowStringArray>
['serie_a', 'ligue_1', 'la_liga', 'bundesliga', 'premier_league']
Length: 5, dtype: str
<ArrowStringArray>
['bundesliga', 'la_liga', 'ligue_1', 'premier_league', 'serie_a']
Length: 5, dtype: str
0   2020-09-21
1   2020-09-27
2   2020-10-04
Name: date, dtype: datetime64[us]
0   2020-09-18
1   2020-09-19
2   2020-09-19
Name: date, dtype: datetime64[us]
datetime64[us] datetime64[us]
['AC Milan', 'Ajaccio', 'Alaves', 'Almeria', 'Angers', 'Arminia Bielefeld', 'Arsenal', 'Aston Villa', 'Atalanta', 'Athletic Club']
['AC Milan', 'Ajaccio', 'Alaves', 'Almeria', 'Angers', 'Arminia Bielefeld', 'Arsenal', 'Aston Villa', 'Atalanta', 'Athletic Club']
team_name               AC Milan
date         2020-09-21 00:00:00
league                   serie_a
year                        2020
Name: 0, dtype: object
      match_id       date   league  year team_name h_a
7088     14122 2020-09-21  serie_a  2020  AC Milan   h
7109     15454 2020-10-04  serie_a  2020  AC Milan   h
7129     15473 2

In [7]:
print(matches['home_result'].value_counts())
print(matches['away_result'].value_counts())

mirror_check = matches[['home_result', 'away_result']].value_counts()
print(mirror_check)

target_map = {'w': 2, 'd': 1, 'l': 0}
matches['target'] = matches['home_result'].map(target_map)

print(matches['target'].isna().sum(), 'unmapped rows — should be 0')
matches = matches.drop(columns=['home_result', 'away_result'])

home_result
w    3835
l    2866
d    2281
Name: count, dtype: int64
away_result
l    3835
w    2866
d    2281
Name: count, dtype: int64
home_result  away_result
w            l              3835
l            w              2866
d            d              2281
Name: count, dtype: int64
0 unmapped rows — should be 0


In [8]:
matches = matches.merge(
    matches_meta[['match_id', 'date']], on='match_id', how='left'
)
matches = matches.sort_values('date').reset_index(drop=True)

# chronological split — no shuffling, no k-fold. last ~20% of matches by date as holdout.
split_date = matches['date'].quantile(0.8)
train = matches[matches['date'] < split_date].copy()
test = matches[matches['date'] >= split_date].copy()

print(split_date)
print(train.shape, test.shape)
print(train['date'].max(), test['date'].min())

2024-05-19 00:00:00
(7174, 39) (1808, 39)
2024-05-18 00:00:00 2024-05-19 00:00:00


In [9]:
# baseline: naive result from trailing xG difference (roll5), no learned model
train_baseline = train.dropna(subset=['home_xG_roll5', 'away_xG_roll5']).copy()
test_baseline = test.dropna(subset=['home_xG_roll5', 'away_xG_roll5']).copy()

def xg_diff_to_result(row, margin=0.15):
    diff = row['home_xG_roll5'] - row['away_xG_roll5']
    if diff > margin:
        return 2  # home win
    elif diff < -margin:
        return 0  # away win
    return 1  # draw

test_baseline['baseline_pred'] = test_baseline.apply(xg_diff_to_result, axis=1)

from sklearn.metrics import accuracy_score, log_loss
print('baseline accuracy:', accuracy_score(test_baseline['target'], test_baseline['baseline_pred']))

baseline accuracy: 0.46811224489795916


In [10]:
pts_map = {'w': 3, 'd': 1, 'l': 0}
teams_df['match_pts'] = teams_df['result'].map(pts_map)
teams_df['pts_to_date'] = teams_df.groupby(['team_name', 'league', 'year'])['match_pts'].transform(
    lambda s: s.shift(1).cumsum()
)

teams_df['rank_to_date'] = teams_df.groupby(['league', 'year', 'matchday'])['pts_to_date'].rank(
    ascending=False, method='first'
)

league_size = teams_df.groupby(['league', 'year'])['team_name'].transform('nunique')
teams_df['tier_to_date'] = pd.cut(
    teams_df['rank_to_date'] / league_size,
    bins=[0, 0.333, 0.667, 1.0],
    labels=['top', 'mid', 'bottom']
)

print(teams_df[teams_df['matchday'].between(8,10)][['team_name','league','year','matchday','pts_to_date','rank_to_date','tier_to_date']].head(15))

    team_name   league  year  matchday  pts_to_date  rank_to_date tier_to_date
7    AC Milan  serie_a  2020         8         17.0           1.0          top
8    AC Milan  serie_a  2020         9         20.0           1.0          top
9    AC Milan  serie_a  2020        10         23.0           1.0          top
45   AC Milan  serie_a  2021         8         19.0           2.0          top
46   AC Milan  serie_a  2021         9         22.0           2.0          top
47   AC Milan  serie_a  2021        10         25.0           1.0          top
83   AC Milan  serie_a  2022         8         14.0           4.0          top
84   AC Milan  serie_a  2022         9         17.0           4.0          top
85   AC Milan  serie_a  2022        10         20.0           3.0          top
121  AC Milan  serie_a  2023         8         18.0           1.0          top
122  AC Milan  serie_a  2023         9         21.0           1.0          top
123  AC Milan  serie_a  2023        10         21.0 

In [11]:
tier_lookup = teams_df[['match_id', 'team_name', 'tier_to_date', 'rank_to_date', 'pts_to_date']].dropna(subset=['match_id'])

home_tier = tier_lookup.rename(columns={
    'team_name': 'home_team', 'tier_to_date': 'home_tier',
    'rank_to_date': 'home_rank', 'pts_to_date': 'home_pts_to_date'
})
away_tier = tier_lookup.rename(columns={
    'team_name': 'away_team', 'tier_to_date': 'away_tier',
    'rank_to_date': 'away_rank', 'pts_to_date': 'away_pts_to_date'
})

matches = matches.merge(home_tier, on=['match_id', 'home_team'], how='left')
matches = matches.merge(away_tier, on=['match_id', 'away_team'], how='left')

matches['rank_gap'] = matches['away_rank'] - matches['home_rank']  # positive = home team ranked higher

print(matches[['home_team','away_team','home_tier','away_tier','rank_gap']].dropna().head(10))
print(matches[['home_tier','away_tier']].isna().sum())

        home_team   away_team home_tier away_tier  rank_gap
9      Strasbourg        Nice    bottom       top     -15.0
11         Angers    Bordeaux       top       mid       7.0
13         Nantes       Nimes       mid       top      -5.0
15          Reims       Lille       mid       mid      -3.0
17       Bordeaux        Lyon       top       top      -1.0
19    Montpellier        Nice    bottom       top     -16.0
20  Saint-Etienne  Strasbourg       mid    bottom      13.0
27         Monaco      Nantes       mid       mid       1.0
29          Lille        Metz       top    bottom      12.0
31          Dijon       Brest    bottom    bottom      -1.0
home_tier    243
away_tier    243
dtype: int64


In [12]:
# clinical_df is match-level per Notebook 04 usage (merged onto teams_df via team_name/league/year/date earlier)
# check whether clinical_df has match_id directly, or needs the same date-normalization + team_name join as before
print('match_id' in clinical_df.columns)
print(clinical_df.columns.tolist()[:15])

True
['h_a', 'xG', 'xGA', 'npxG', 'npxGA', 'ppda', 'ppda_allowed', 'deep', 'deep_allowed', 'scored', 'missed', 'xpts', 'result', 'date', 'wins']


In [13]:
xgd_lookup = clinical_df[['match_id', 'team_name', 'match_xGD']].drop_duplicates(subset=['match_id', 'team_name'])
teams_df = teams_df.merge(xgd_lookup, on=['match_id', 'team_name'], how='left')
print(teams_df['match_xGD'].isna().sum(), 'unmatched — should be 0 or small')

0 unmatched — should be 0 or small


In [14]:
grouped = teams_df.groupby(['team_name', 'league', 'year'])

teams_df['rolling_pts_5'] = grouped['match_pts'].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).sum()
)
teams_df['rolling_xGD_5'] = grouped['match_xGD'].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).sum()
)
teams_df['form_xGD_divergence'] = teams_df['rolling_pts_5'] - teams_df['rolling_xGD_5']

print(teams_df[teams_df['matchday']==1][['rolling_pts_5','rolling_xGD_5','form_xGD_divergence']].isna().all())

rolling_pts_5          True
rolling_xGD_5          True
form_xGD_divergence    True
dtype: bool


In [15]:
form_lookup = teams_df[['match_id', 'team_name', 'rolling_pts_5', 'rolling_xGD_5', 'form_xGD_divergence']]

home_form = form_lookup.rename(columns={
    'team_name': 'home_team',
    'rolling_pts_5': 'home_rolling_pts_5',
    'rolling_xGD_5': 'home_rolling_xGD_5',
    'form_xGD_divergence': 'home_form_xGD_divergence'
})
away_form = form_lookup.rename(columns={
    'team_name': 'away_team',
    'rolling_pts_5': 'away_rolling_pts_5',
    'rolling_xGD_5': 'away_rolling_xGD_5',
    'form_xGD_divergence': 'away_form_xGD_divergence'
})

matches = matches.merge(home_form, on=['match_id', 'home_team'], how='left')
matches = matches.merge(away_form, on=['match_id', 'away_team'], how='left')

print(matches.shape)
print(matches[[c for c in matches.columns if 'form' in c]].isna().sum())

(8982, 52)
home_form_xGD_divergence    1219
away_form_xGD_divergence    1211
dtype: int64


In [16]:
feature_cols_final = (
    [f'home_{m}_roll5' for m in roll_metrics] + [f'away_{m}_roll5' for m in roll_metrics] +
    [f'home_{m}_expanding' for m in roll_metrics] + [f'away_{m}_expanding' for m in roll_metrics] +
    ['home_rank', 'away_rank', 'rank_gap',
     'home_rolling_pts_5', 'away_rolling_pts_5',
     'home_form_xGD_divergence', 'away_form_xGD_divergence']
)

matches = matches.sort_values('date').reset_index(drop=True)
split_date = matches['date'].quantile(0.8)
train = matches[matches['date'] < split_date].copy()
test = matches[matches['date'] >= split_date].copy()

# drop rows with NaN in any required feature (early-season rows before roll5/expanding windows fill in)
train_clean = train.dropna(subset=feature_cols_final + ['target'])
test_clean = test.dropna(subset=feature_cols_final + ['target'])

print(train.shape, train_clean.shape)
print(test.shape, test_clean.shape)

(7174, 52) (5900, 52)
(1808, 52) (1471, 52)


In [17]:
# isolate which specific columns are driving the drop
for col in feature_cols_final:
    n_na = train[col].isna().sum()
    if n_na > 0:
        print(col, n_na)

home_ppda_value_roll5 1054
home_ppda_allowed_value_roll5 1047
home_scored_roll5 979
home_missed_roll5 979
home_xG_roll5 979
home_xGA_roll5 979
home_deep_roll5 979
home_deep_allowed_roll5 979
away_ppda_value_roll5 1044
away_ppda_allowed_value_roll5 1044
away_scored_roll5 971
away_missed_roll5 971
away_xG_roll5 971
away_xGA_roll5 971
away_deep_roll5 971
away_deep_allowed_roll5 971
home_ppda_value_expanding 591
home_ppda_allowed_value_expanding 593
home_scored_expanding 588
home_missed_expanding 588
home_xG_expanding 588
home_xGA_expanding 588
home_deep_expanding 588
home_deep_allowed_expanding 588
away_ppda_value_expanding 584
away_ppda_allowed_value_expanding 584
away_scored_expanding 582
away_missed_expanding 582
away_xG_expanding 582
away_xGA_expanding 582
away_deep_expanding 582
away_deep_allowed_expanding 582
home_rank 195
away_rank 195
rank_gap 206
home_rolling_pts_5 979
away_rolling_pts_5 971
home_form_xGD_divergence 979
away_form_xGD_divergence 971


In [18]:
for col in ['home_rank', 'away_rank', 'rank_gap']:
    print(col, train[col].isna().sum())

home_rank 195
away_rank 195
rank_gap 206


In [19]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, log_loss, classification_report

X_train = train_clean[feature_cols_final]
y_train = train_clean['target']
X_test = test_clean[feature_cols_final]
y_test = test_clean['target']

model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print('accuracy:', accuracy_score(y_test, y_pred))
print('log_loss:', log_loss(y_test, y_proba))
print(classification_report(y_test, y_pred, target_names=['away_win','draw','home_win']))

accuracy: 0.5003399048266486
log_loss: 1.0132564306259155
              precision    recall  f1-score   support

    away_win       0.51      0.51      0.51       486
        draw       0.27      0.09      0.13       362
    home_win       0.52      0.74      0.61       623

    accuracy                           0.50      1471
   macro avg       0.44      0.44      0.42      1471
weighted avg       0.46      0.50      0.46      1471



In [20]:
def xg_diff_to_proba(row, margin=0.15, confidence=0.6):
    diff = row['home_xG_roll5'] - row['away_xG_roll5']
    if diff > margin:
        return [ (1-confidence)/2, (1-confidence)/2, confidence ]  # [away, draw, home]
    elif diff < -margin:
        return [ confidence, (1-confidence)/2, (1-confidence)/2 ]
    else:
        return [ (1-confidence)/2, confidence, (1-confidence)/2 ]

baseline_proba = np.array(test_baseline.apply(xg_diff_to_proba, axis=1).tolist())
print('baseline log_loss:', log_loss(test_baseline['target'], baseline_proba))

baseline log_loss: 1.0951640477131868


In [21]:
# empirical class distribution within each bucket, computed from TRAIN only (no leakage into test)
def xg_diff_bucket(row, margin=0.15):
    diff = row['home_xG_roll5'] - row['away_xG_roll5']
    if diff > margin:
        return 'home_lean'
    elif diff < -margin:
        return 'away_lean'
    return 'neutral'

train_baseline['bucket'] = train_baseline.apply(xg_diff_bucket, axis=1)
test_baseline['bucket'] = test_baseline.apply(xg_diff_bucket, axis=1)

# empirical P(target | bucket) from train, in class order [away=0, draw=1, home=2]
bucket_probs = (
    train_baseline.groupby('bucket')['target']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=[0, 1, 2], fill_value=0)  # ensure all 3 classes present even if a bucket never saw one
)
print(bucket_probs)

baseline_proba = test_baseline['bucket'].map(bucket_probs.to_dict('index')).apply(pd.Series).values
print('baseline log_loss (calibrated):', log_loss(test_baseline['target'], baseline_proba))

target            0         1         2
bucket                                 
away_lean  0.439850  0.262030  0.298120
home_lean  0.196560  0.227273  0.576167
neutral    0.268698  0.284395  0.446907
baseline log_loss (calibrated): 1.0372462675078762


In [22]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model_weighted = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

model_weighted.fit(X_train, y_train, sample_weight=sample_weights)

y_pred_w = model_weighted.predict(X_test)
y_proba_w = model_weighted.predict_proba(X_test)

print('accuracy:', accuracy_score(y_test, y_pred_w))
print('log_loss:', log_loss(y_test, y_proba_w))
print(classification_report(y_test, y_pred_w, target_names=['away_win','draw','home_win']))

accuracy: 0.49014276002719237
log_loss: 1.0282262563705444
              precision    recall  f1-score   support

    away_win       0.51      0.52      0.52       486
        draw       0.30      0.27      0.29       362
    home_win       0.57      0.59      0.58       623

    accuracy                           0.49      1471
   macro avg       0.46      0.46      0.46      1471
weighted avg       0.48      0.49      0.49      1471



In [23]:
importance_unweighted = pd.Series(model.feature_importances_, index=feature_cols_final).sort_values(ascending=False)
importance_weighted = pd.Series(model_weighted.feature_importances_, index=feature_cols_final).sort_values(ascending=False)

importance_compare = pd.DataFrame({
    'unweighted': importance_unweighted,
    'weighted': importance_weighted
}).sort_values('weighted', ascending=False)

print(importance_compare.head(15))

                                   unweighted  weighted
rank_gap                             0.133489  0.114273
away_deep_expanding                  0.038144  0.037818
home_xG_expanding                    0.031953  0.033145
home_deep_expanding                  0.032637  0.032250
away_xG_expanding                    0.032302  0.032019
home_ppda_allowed_value_expanding    0.028888  0.029165
away_ppda_allowed_value_expanding    0.026180  0.026513
away_rank                            0.025424  0.026272
home_rank                            0.024106  0.025264
away_deep_allowed_expanding          0.022759  0.023269
home_deep_allowed_expanding          0.021659  0.023018
home_missed_expanding                0.023209  0.022929
home_form_xGD_divergence             0.022660  0.022877
away_xG_roll5                        0.020934  0.022687
home_xGA_expanding                   0.022828  0.022493


In [24]:
from sklearn.model_selection import ParameterGrid

train_clean_sorted = train_clean.sort_values('date')
val_split = train_clean_sorted['date'].quantile(0.85)
tr = train_clean_sorted[train_clean_sorted['date'] < val_split]
val = train_clean_sorted[train_clean_sorted['date'] >= val_split]

X_tr, y_tr = tr[feature_cols_final], tr['target']
X_val, y_val = val[feature_cols_final], val['target']
sw_tr = compute_sample_weight(class_weight='balanced', y=y_tr)

param_grid = {
    'max_depth': [3, 4, 6],
    'learning_rate': [0.03, 0.05, 0.1],
    'n_estimators': [200, 400]
}

results = []
for params in ParameterGrid(param_grid):
    m = xgb.XGBClassifier(
        objective='multi:softprob', num_class=3,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='mlogloss', **params
    )
    m.fit(X_tr, y_tr, sample_weight=sw_tr)
    proba = m.predict_proba(X_val)
    ll = log_loss(y_val, proba)
    results.append({**params, 'val_log_loss': ll})

results_df = pd.DataFrame(results).sort_values('val_log_loss')
print(results_df.head(10))

    learning_rate  max_depth  n_estimators  val_log_loss
0            0.03          3           200      1.000019
4            0.03          6           200      1.001759
2            0.03          4           200      1.002275
1            0.03          3           400      1.003680
6            0.05          3           200      1.004197
8            0.05          4           200      1.006655
3            0.03          4           400      1.006772
10           0.05          6           200      1.007994
5            0.03          6           400      1.010291
7            0.05          3           400      1.010374


In [25]:
final_model = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3,
    max_depth=3, learning_rate=0.03, n_estimators=200,
    subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='mlogloss'
)
sample_weights_full = compute_sample_weight(class_weight='balanced', y=y_train)
final_model.fit(X_train, y_train, sample_weight=sample_weights_full)

y_pred_final = final_model.predict(X_test)
y_proba_final = final_model.predict_proba(X_test)

print('accuracy:', accuracy_score(y_test, y_pred_final))
print('log_loss:', log_loss(y_test, y_proba_final))
print(classification_report(y_test, y_pred_final, target_names=['away_win','draw','home_win']))

accuracy: 0.5064581917063222
log_loss: 1.011678695678711
              precision    recall  f1-score   support

    away_win       0.52      0.52      0.52       486
        draw       0.31      0.30      0.30       362
    home_win       0.60      0.62      0.61       623

    accuracy                           0.51      1471
   macro avg       0.48      0.48      0.48      1471
weighted avg       0.50      0.51      0.50      1471



In [26]:
# one-hot encode tier, since XGBoost handles categoricals better as explicit dummies here (no ordinal assumption)
matches = pd.get_dummies(matches, columns=['home_tier', 'away_tier'], prefix=['home_tier', 'away_tier'])
tier_dummy_cols = [c for c in matches.columns if c.startswith('home_tier_') or c.startswith('away_tier_')]
print(tier_dummy_cols)

['home_tier_top', 'home_tier_mid', 'home_tier_bottom', 'away_tier_top', 'away_tier_mid', 'away_tier_bottom']


In [27]:
shots_df['is_goal'] = (shots_df['result'] == 'Goal').astype(int)

# per-match, per-team counter-attack share and conversion (this match's own numbers — will be shifted before use)
match_counter = shots_df.groupby(['match_id', 'team_name']).agg(
    total_shots=('counter_score', 'count'),
    counter_shots=('counter_score', lambda x: (x >= 2).sum()),
    counter_goals=('is_goal', lambda x: x[shots_df.loc[x.index, 'counter_score'] >= 2].sum())
).reset_index()

match_counter['counter_share'] = match_counter['counter_shots'] / match_counter['total_shots'].replace(0, np.nan)

# attach match date + team's own matchday sequence so we can roll it in team-date order
match_counter = match_counter.merge(
    teams_df[['match_id', 'team_name', 'date', 'league', 'year']].drop_duplicates(),
    on=['match_id', 'team_name'], how='left'
)
match_counter = match_counter.sort_values(['team_name', 'date'])

match_counter['counter_share_roll5'] = match_counter.groupby(['team_name', 'league', 'year'])['counter_share'].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean()
)

print(match_counter[['team_name','date','counter_share','counter_share_roll5']].head(10))

     team_name       date  counter_share  counter_share_roll5
106   AC Milan 2020-09-21       0.000000                  NaN
122   AC Milan 2020-09-27       0.000000                  NaN
2286  AC Milan 2020-10-04       0.000000                  NaN
2292  AC Milan 2020-10-17       0.000000                  NaN
2324  AC Milan 2020-10-26       0.000000                  NaN
2346  AC Milan 2020-11-01       0.000000             0.000000
2362  AC Milan 2020-11-08       0.000000             0.000000
2378  AC Milan 2020-11-22       0.066667             0.000000
2402  AC Milan 2020-11-29       0.000000             0.013333
2422  AC Milan 2020-12-06       0.000000             0.013333


In [28]:
shots_df['is_goal'] = (shots_df['result'] == 'Goal').astype(int)

total_shots = shots_df.groupby(['match_id', 'team_name']).size().rename('total_shots')
counter_shots = shots_df[shots_df['counter_score'] >= 2].groupby(['match_id', 'team_name']).size().rename('counter_shots')

match_counter = pd.concat([total_shots, counter_shots], axis=1).reset_index()
match_counter['counter_shots'] = match_counter['counter_shots'].fillna(0)
match_counter['counter_share'] = match_counter['counter_shots'] / match_counter['total_shots']

match_counter = match_counter.merge(
    teams_df[['match_id', 'team_name', 'date', 'league', 'year']].drop_duplicates(),
    on=['match_id', 'team_name'], how='left'
)
match_counter = match_counter.sort_values(['team_name', 'date'])

match_counter['counter_share_roll5'] = match_counter.groupby(['team_name', 'league', 'year'])['counter_share'].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean()
)

print(match_counter[['team_name','date','counter_share','counter_share_roll5']].head(10))

     team_name       date  counter_share  counter_share_roll5
106   AC Milan 2020-09-21       0.000000                  NaN
122   AC Milan 2020-09-27       0.000000                  NaN
2286  AC Milan 2020-10-04       0.000000                  NaN
2292  AC Milan 2020-10-17       0.000000                  NaN
2324  AC Milan 2020-10-26       0.000000                  NaN
2346  AC Milan 2020-11-01       0.000000             0.000000
2362  AC Milan 2020-11-08       0.000000             0.000000
2378  AC Milan 2020-11-22       0.066667             0.000000
2402  AC Milan 2020-11-29       0.000000             0.013333
2422  AC Milan 2020-12-06       0.000000             0.013333


In [29]:
print(shots_df['counter_score'].value_counts())
print((shots_df['counter_score'] >= 2).sum(), 'total counter shots across all data')

counter_score
0    165588
1     57155
2      1933
Name: count, dtype: int64
1933 total counter shots across all data


In [30]:
# tier dummies (from the get_dummies step earlier)
# already merged into `matches` in-place via pd.get_dummies, no further join needed there

# counter-attack rolling feature — merge home/away onto matches
counter_lookup = match_counter[['match_id', 'team_name', 'counter_share_roll5']]

home_counter = counter_lookup.rename(columns={'team_name': 'home_team', 'counter_share_roll5': 'home_counter_share_roll5'})
away_counter = counter_lookup.rename(columns={'team_name': 'away_team', 'counter_share_roll5': 'away_counter_share_roll5'})

matches = matches.merge(home_counter, on=['match_id', 'home_team'], how='left')
matches = matches.merge(away_counter, on=['match_id', 'away_team'], how='left')

print(matches[['home_counter_share_roll5', 'away_counter_share_roll5']].isna().sum())

home_counter_share_roll5    1222
away_counter_share_roll5    1216
dtype: int64


In [31]:
# players_enriched has career trajectory labels per player-season (per Notebook 05)
players_df = pd.read_csv('../data/processed/players_enriched.csv')
print(players_df.columns.tolist())
print(players_df['trajectory'].value_counts() if 'trajectory' in players_df.columns else 'column name differs — check')

['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'league', 'year', 'primary_position', 'per90_reliable', 'npxG_per90', 'goals_per90', 'xGChain_per90', 'xA_per90', 'key_passes_per90', 'primary_position_hierarchy', 'primary_position_minutes']
column name differs — check


In [32]:
# players_enriched has career trajectory labels per player-season (per Notebook 05)
players_df = pd.read_csv('../data/processed/players_enriched.csv')
print(players_df.columns.tolist())
print(players_df['trajectory'].value_counts() if 'trajectory' in players_df.columns else 'column name differs — check')

['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'league', 'year', 'primary_position', 'per90_reliable', 'npxG_per90', 'goals_per90', 'xGChain_per90', 'xA_per90', 'key_passes_per90', 'primary_position_hierarchy', 'primary_position_minutes']
column name differs — check


In [33]:
player_year_agg = players_df.groupby(['id', 'player_name', 'year']).agg(
    total_time=('time', 'sum'),
    total_npxG=('npxG', 'sum'),
    total_goals=('goals', 'sum'),
    clubs=('team_title', lambda x: ' → '.join(x))
).reset_index()
player_year_agg['npxG_per90_combined'] = (player_year_agg['total_npxG'] / player_year_agg['total_time'] * 90).round(3)
player_year_agg['per90_reliable_combined'] = player_year_agg['total_time'] >= 450

In [34]:
trajectory_pool_v2 = player_year_agg[player_year_agg['per90_reliable_combined']].sort_values(['id', 'year']).copy()

position_lookup = players_df.groupby(['id', 'year'])['primary_position_minutes'].first().reset_index()
trajectory_pool_v2 = trajectory_pool_v2.merge(position_lookup, on=['id', 'year'], how='left')
trajectory_pool_v2 = trajectory_pool_v2[trajectory_pool_v2['primary_position_minutes'] != 'GK']

def classify_trajectory_v2(group):
    group = group.sort_values('year')
    years = group['year'].values
    if len(group) < 3:
        return 'insufficient_data'
    year_gaps = np.diff(years)
    if not (year_gaps == 1).all():
        return 'non_consecutive_seasons'
    diffs = group['npxG_per90_combined'].diff().dropna().values
    last_three = diffs[-3:] if len(diffs) >= 3 else diffs
    if (last_three > 0).all():
        return 'ascending'
    elif (last_three < 0).all():
        return 'declining'
    else:
        return 'established'

trajectories_v2 = trajectory_pool_v2.groupby('id').apply(classify_trajectory_v2, include_groups=False)
print(trajectories_v2.value_counts())

insufficient_data          2007
established                1072
non_consecutive_seasons     271
ascending                   131
declining                   128
Name: count, dtype: int64


In [35]:
def classify_trajectory_at_cutoff(player_years_df, cutoff_year):
    """
    For a single player's year-by-year npxG_per90_combined history, classify their trajectory
    using only seasons up to and including cutoff_year (no future seasons visible).
    """
    hist = player_years_df[player_years_df['year'] <= cutoff_year].sort_values('year')
    if len(hist) < 3:
        return 'insufficient_data'

    years = hist['year'].values
    # only consider the trailing consecutive run ending at cutoff_year
    if years[-1] != cutoff_year:
        return 'insufficient_data'  # player didn't play in cutoff_year itself

    year_gaps = np.diff(years)
    if not (year_gaps == 1).all():
        return 'non_consecutive_seasons'

    diffs = hist['npxG_per90_combined'].diff().dropna().values
    last_three = diffs[-3:] if len(diffs) >= 3 else diffs

    if (last_three > 0).all():
        return 'ascending'
    elif (last_three < 0).all():
        return 'declining'
    else:
        return 'established'

# reuse the reliable, GK-excluded pool built earlier
trajectory_pool_v2 = player_year_agg[player_year_agg['per90_reliable_combined']].sort_values(['id', 'year']).copy()
position_lookup = players_df.groupby(['id', 'year'])['primary_position_minutes'].first().reset_index()
trajectory_pool_v2 = trajectory_pool_v2.merge(position_lookup, on=['id', 'year'], how='left')
trajectory_pool_v2 = trajectory_pool_v2[trajectory_pool_v2['primary_position_minutes'] != 'GK']

all_years = sorted(trajectory_pool_v2['year'].unique())
records = []
for pid, group in trajectory_pool_v2.groupby('id'):
    for cutoff in all_years:
        label = classify_trajectory_at_cutoff(group, cutoff)
        if label != 'insufficient_data':  # skip years the player has no data for
            records.append({'id': pid, 'cutoff_year': cutoff, 'trajectory_label': label})

traj_by_cutoff = pd.DataFrame(records)
print(traj_by_cutoff['trajectory_label'].value_counts())
print(traj_by_cutoff.head(10))

trajectory_label
established                2177
ascending                   353
non_consecutive_seasons     345
declining                   296
Name: count, dtype: int64
   id  cutoff_year         trajectory_label
0   3         2023  non_consecutive_seasons
1   3         2024  non_consecutive_seasons
2  22         2022              established
3  22         2023              established
4  23         2022              established
5  28         2024                ascending
6  40         2022              established
7  40         2023              established
8  40         2024              established
9  52         2022              established


In [36]:
player_team_year = players_df[['id', 'team_title', 'year']].drop_duplicates()

# join each player's team-in-year-Y to their trajectory-as-of-year-Y
team_traj = player_team_year.merge(
    traj_by_cutoff, left_on=['id', 'year'], right_on=['id', 'cutoff_year'], how='inner'
)

team_traj_composition = (
    team_traj.groupby(['team_title', 'year'])['trajectory_label']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['ascending', 'declining', 'established', 'non_consecutive_seasons']:
    if col not in team_traj_composition.columns:
        team_traj_composition[col] = 0.0

team_traj_composition = team_traj_composition.rename(columns={
    'ascending': 'pct_ascending', 'declining': 'pct_declining', 'established': 'pct_established'
})

# lag: team's composition AS OF year Y becomes a feature for year Y+1's matches
team_traj_composition['year'] = team_traj_composition['year'] + 1
team_traj_composition = team_traj_composition.rename(columns={'team_title': 'team_name'})[
    ['team_name', 'year', 'pct_ascending', 'pct_declining', 'pct_established']
]

print(team_traj_composition.head(10))

trajectory_label            team_name  year  pct_ascending  pct_declining  \
0                            AC Milan  2023       0.062500       0.062500   
1                            AC Milan  2024       0.000000       0.000000   
2                            AC Milan  2025       0.058824       0.000000   
3                    AC Milan,Bologna  2025       0.000000       0.000000   
4                 AC Milan,Fiorentina  2025       0.000000       0.000000   
5                       AC Milan,Roma  2025       0.000000       0.000000   
6                             Ajaccio  2023       0.000000       0.000000   
7                              Alaves  2024       0.000000       0.000000   
8                              Alaves  2025       0.000000       0.142857   
9                       Alaves,Getafe  2025       0.000000       0.000000   

trajectory_label  pct_established  
0                        0.875000  
1                        0.764706  
2                        0.823529  
3       

In [37]:
player_team_year = players_df[['id', 'team_title', 'year']].drop_duplicates().copy()

# split comma-joined multi-club strings into one row per actual club
player_team_year['team_title'] = player_team_year['team_title'].str.split(',')
player_team_year = player_team_year.explode('team_title')
player_team_year['team_title'] = player_team_year['team_title'].str.strip()

print(player_team_year['team_title'].nunique(), 'unique teams after split')
print(sorted(player_team_year['team_title'].unique())[:15])

132 unique teams after split
['AC Milan', 'Ajaccio', 'Alaves', 'Almeria', 'Angers', 'Arminia Bielefeld', 'Arsenal', 'Aston Villa', 'Atalanta', 'Athletic Club', 'Atletico Madrid', 'Augsburg', 'Auxerre', 'Barcelona', 'Bayer Leverkusen']


In [38]:
print(sorted(players_df['year'].unique()))

[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [39]:
team_traj = player_team_year.merge(
    traj_by_cutoff, left_on=['id', 'year'], right_on=['id', 'cutoff_year'], how='inner'
)

team_traj_composition = (
    team_traj.groupby(['team_title', 'year'])['trajectory_label']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['ascending', 'declining', 'established', 'non_consecutive_seasons']:
    if col not in team_traj_composition.columns:
        team_traj_composition[col] = 0.0

team_traj_composition = team_traj_composition.rename(columns={
    'ascending': 'pct_ascending', 'declining': 'pct_declining', 'established': 'pct_established'
})

team_traj_composition['year'] = team_traj_composition['year'] + 1
team_traj_composition = team_traj_composition.rename(columns={'team_title': 'team_name'})[
    ['team_name', 'year', 'pct_ascending', 'pct_declining', 'pct_established']
]

print(team_traj_composition.head(10))

trajectory_label team_name  year  pct_ascending  pct_declining  \
0                 AC Milan  2023       0.062500       0.062500   
1                 AC Milan  2024       0.000000       0.000000   
2                 AC Milan  2025       0.047619       0.000000   
3                  Ajaccio  2023       0.000000       0.000000   
4                   Alaves  2024       0.000000       0.000000   
5                   Alaves  2025       0.000000       0.111111   
6                  Almeria  2023       0.250000       0.250000   
7                  Almeria  2024       0.000000       0.250000   
8                   Angers  2023       0.285714       0.142857   
9                   Angers  2025       0.000000       0.000000   

trajectory_label  pct_established  
0                        0.875000  
1                        0.764706  
2                        0.809524  
3                        1.000000  
4                        0.400000  
5                        0.333333  
6                    

In [40]:
print(set(player_team_year['team_title'].unique()) - set(teams_df['team_name'].unique()))

set()


In [41]:
home_traj = team_traj_composition.rename(columns={
    'team_name': 'home_team', 'pct_ascending': 'home_pct_ascending',
    'pct_declining': 'home_pct_declining', 'pct_established': 'home_pct_established'
})
away_traj = team_traj_composition.rename(columns={
    'team_name': 'away_team', 'pct_ascending': 'away_pct_ascending',
    'pct_declining': 'away_pct_declining', 'pct_established': 'away_pct_established'
})

matches = matches.merge(home_traj, on=['home_team', 'year'], how='left')
matches = matches.merge(away_traj, on=['away_team', 'year'], how='left')

traj_cols = ['home_pct_ascending', 'home_pct_declining', 'home_pct_established',
             'away_pct_ascending', 'away_pct_declining', 'away_pct_established']
print(matches[traj_cols].isna().sum())

home_pct_ascending      6024
home_pct_declining      6024
home_pct_established    6024
away_pct_ascending      6024
away_pct_declining      6024
away_pct_established    6024
dtype: int64


In [42]:
feature_cols_v2 = feature_cols_final + tier_dummy_cols + [
    'home_counter_share_roll5', 'away_counter_share_roll5'
] + traj_cols

train = matches[matches['date'] < split_date].copy()
test = matches[matches['date'] >= split_date].copy()

train_clean_v2 = train.dropna(subset=feature_cols_v2 + ['target'])
test_clean_v2 = test.dropna(subset=feature_cols_v2 + ['target'])

print(train.shape, train_clean_v2.shape, test.shape, test_clean_v2.shape)

X_train_v2 = train_clean_v2[feature_cols_v2]
y_train_v2 = train_clean_v2['target']
X_test_v2 = test_clean_v2[feature_cols_v2]
y_test_v2 = test_clean_v2['target']

sw_v2 = compute_sample_weight(class_weight='balanced', y=y_train_v2)

model_v2 = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3,
    max_depth=3, learning_rate=0.03, n_estimators=200,
    subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='mlogloss'
)
model_v2.fit(X_train_v2, y_train_v2, sample_weight=sw_v2)

y_pred_v2 = model_v2.predict(X_test_v2)
y_proba_v2 = model_v2.predict_proba(X_test_v2)

print('accuracy:', accuracy_score(y_test_v2, y_pred_v2))
print('log_loss:', log_loss(y_test_v2, y_proba_v2))
print(classification_report(y_test_v2, y_pred_v2, target_names=['away_win','draw','home_win']))

(7174, 64) (971, 64) (1808, 64) (1025, 64)
accuracy: 0.4741463414634146
log_loss: 1.0431660413742065
              precision    recall  f1-score   support

    away_win       0.46      0.42      0.44       323
        draw       0.28      0.30      0.29       262
    home_win       0.60      0.61      0.61       440

    accuracy                           0.47      1025
   macro avg       0.45      0.45      0.45      1025
weighted avg       0.48      0.47      0.47      1025



In [43]:
feature_cols_v3 = feature_cols_final + tier_dummy_cols + ['home_counter_share_roll5', 'away_counter_share_roll5']

train_clean_v3 = train.dropna(subset=feature_cols_v3 + ['target'])
test_clean_v3 = test.dropna(subset=feature_cols_v3 + ['target'])

print(train_clean_v3.shape, test_clean_v3.shape)

X_train_v3 = train_clean_v3[feature_cols_v3]
y_train_v3 = train_clean_v3['target']
X_test_v3 = test_clean_v3[feature_cols_v3]
y_test_v3 = test_clean_v3['target']

sw_v3 = compute_sample_weight(class_weight='balanced', y=y_train_v3)

model_v3 = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3,
    max_depth=3, learning_rate=0.03, n_estimators=200,
    subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='mlogloss'
)
model_v3.fit(X_train_v3, y_train_v3, sample_weight=sw_v3)

y_pred_v3 = model_v3.predict(X_test_v3)
y_proba_v3 = model_v3.predict_proba(X_test_v3)

print('accuracy:', accuracy_score(y_test_v3, y_pred_v3))
print('log_loss:', log_loss(y_test_v3, y_proba_v3))
print(classification_report(y_test_v3, y_pred_v3, target_names=['away_win','draw','home_win']))

(5896, 64) (1469, 64)
accuracy: 0.505786249149081
log_loss: 1.01206374168396
              precision    recall  f1-score   support

    away_win       0.51      0.53      0.52       486
        draw       0.30      0.28      0.29       361
    home_win       0.60      0.62      0.61       622

    accuracy                           0.51      1469
   macro avg       0.47      0.48      0.47      1469
weighted avg       0.50      0.51      0.50      1469



In [44]:
# unordered pair key so Arsenal-vs-Chelsea and Chelsea-vs-Arsenal share history
matches['pair_key'] = matches.apply(lambda r: tuple(sorted([r['home_team'], r['away_team']])), axis=1)
matches = matches.sort_values('date').reset_index(drop=True)

def compute_h2h_features(matches):
    records = []
    history = {}  # pair_key -> list of past match dicts

    for idx, row in matches.iterrows():
        key = row['pair_key']
        past = history.get(key, [])

        if len(past) == 0:
            records.append({'match_id': row['match_id'], 'h2h_matches': 0,
                             'h2h_home_win_rate': np.nan, 'h2h_draw_rate': np.nan,
                             'h2h_away_win_rate': np.nan, 'h2h_xg_diff': np.nan})
        else:
            # re-express each past meeting from the CURRENT home team's perspective
            home_team = row['home_team']
            home_wins = sum(1 for m in past if (m['winner'] == home_team))
            draws = sum(1 for m in past if m['winner'] == 'draw')
            away_wins = len(past) - home_wins - draws
            avg_xg_for = np.mean([m['xg_for'][home_team] for m in past if home_team in m['xg_for']])
            avg_xg_against = np.mean([m['xg_for'][m['away_team_orig'] if m['home_team_orig']==home_team else m['home_team_orig']] for m in past])

            records.append({
                'match_id': row['match_id'],
                'h2h_matches': len(past),
                'h2h_home_win_rate': home_wins / len(past),
                'h2h_draw_rate': draws / len(past),
                'h2h_away_win_rate': away_wins / len(past),
                'h2h_xg_diff': avg_xg_for - avg_xg_against
            })

        # append this match to history AFTER using it (so it's available for future matches only)
        winner = row['home_team'] if row['target'] == 2 else (row['away_team'] if row['target'] == 0 else 'draw')
        past.append({
            'winner': winner,
            'home_team_orig': row['home_team'],
            'away_team_orig': row['away_team'],
            'xg_for': {row['home_team']: row.get('home_xG_roll5', np.nan), row['away_team']: row.get('away_xG_roll5', np.nan)}
        })
        history[key] = past

    return pd.DataFrame(records)

In [45]:
matches = matches.merge(
    matches_meta[['match_id', 'home_xG', 'away_xG']], on='match_id', how='left'
)
print(matches[['home_xG', 'away_xG']].isna().sum())

home_xG    0
away_xG    0
dtype: int64


In [46]:
matches['pair_key'] = matches.apply(lambda r: tuple(sorted([r['home_team'], r['away_team']])), axis=1)
matches = matches.sort_values('date').reset_index(drop=True)

def compute_h2h_features(matches):
    records = []
    history = {}  # pair_key -> list of past meeting dicts

    for idx, row in matches.iterrows():
        key = row['pair_key']
        past = history.get(key, [])

        if len(past) == 0:
            records.append({'match_id': row['match_id'], 'h2h_matches': 0,
                             'h2h_this_home_win_rate': np.nan, 'h2h_draw_rate': np.nan,
                             'h2h_this_away_win_rate': np.nan, 'h2h_xg_diff': np.nan})
        else:
            this_home = row['home_team']
            this_away = row['away_team']

            this_home_wins = sum(1 for m in past if m['winner'] == this_home)
            draws = sum(1 for m in past if m['winner'] == 'draw')
            this_away_wins = len(past) - this_home_wins - draws

            avg_xg_this_home = np.mean([m['xg'][this_home] for m in past])
            avg_xg_this_away = np.mean([m['xg'][this_away] for m in past])

            records.append({
                'match_id': row['match_id'],
                'h2h_matches': len(past),
                'h2h_this_home_win_rate': this_home_wins / len(past),
                'h2h_draw_rate': draws / len(past),
                'h2h_this_away_win_rate': this_away_wins / len(past),
                'h2h_xg_diff': avg_xg_this_home - avg_xg_this_away
            })

        winner = row['home_team'] if row['target'] == 2 else (row['away_team'] if row['target'] == 0 else 'draw')
        past.append({
            'winner': winner,
            'xg': {row['home_team']: row['home_xG'], row['away_team']: row['away_xG']}
        })
        history[key] = past

    return pd.DataFrame(records)

h2h_features = compute_h2h_features(matches)
print(h2h_features['h2h_matches'].value_counts().sort_index())
print(h2h_features.head(10))

h2h_matches
0    1557
1    1557
2    1122
3    1122
4     864
5     864
6     582
7     582
8     366
9     366
Name: count, dtype: int64
   match_id  h2h_matches  h2h_this_home_win_rate  h2h_draw_rate  \
0     13977            0                     NaN            NaN   
1     13979            0                     NaN            NaN   
2     13978            0                     NaN            NaN   
3     13980            0                     NaN            NaN   
4     13982            0                     NaN            NaN   
5     13984            0                     NaN            NaN   
6     13985            0                     NaN            NaN   
7     13990            0                     NaN            NaN   
8     13994            0                     NaN            NaN   
9     13996            0                     NaN            NaN   

   h2h_this_away_win_rate  h2h_xg_diff  
0                     NaN          NaN  
1                     NaN          NaN  
2

In [47]:
matches = matches.merge(h2h_features, on='match_id', how='left')

h2h_cols = ['h2h_matches', 'h2h_this_home_win_rate', 'h2h_draw_rate', 'h2h_this_away_win_rate', 'h2h_xg_diff']

# impute: 0 prior meetings -> neutral values, not dropped rows
matches['h2h_this_home_win_rate'] = matches['h2h_this_home_win_rate'].fillna(0.333)
matches['h2h_draw_rate'] = matches['h2h_draw_rate'].fillna(0.333)
matches['h2h_this_away_win_rate'] = matches['h2h_this_away_win_rate'].fillna(0.333)
matches['h2h_xg_diff'] = matches['h2h_xg_diff'].fillna(0.0)
# h2h_matches itself stays as 0 — that's already meaningful (no imputation needed)

feature_cols_h2h = feature_cols_final + h2h_cols

train = matches[matches['date'] < split_date].copy()
test = matches[matches['date'] >= split_date].copy()

train_clean_h2h = train.dropna(subset=feature_cols_h2h + ['target'])
test_clean_h2h = test.dropna(subset=feature_cols_h2h + ['target'])

print(train_clean_h2h.shape, test_clean_h2h.shape)

X_train_h2h = train_clean_h2h[feature_cols_h2h]
y_train_h2h = train_clean_h2h['target']
X_test_h2h = test_clean_h2h[feature_cols_h2h]
y_test_h2h = test_clean_h2h['target']

sw_h2h = compute_sample_weight(class_weight='balanced', y=y_train_h2h)

model_h2h = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3,
    max_depth=3, learning_rate=0.03, n_estimators=200,
    subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='mlogloss'
)
model_h2h.fit(X_train_h2h, y_train_h2h, sample_weight=sw_h2h)

y_pred_h2h = model_h2h.predict(X_test_h2h)
y_proba_h2h = model_h2h.predict_proba(X_test_h2h)

print('accuracy:', accuracy_score(y_test_h2h, y_pred_h2h))
print('log_loss:', log_loss(y_test_h2h, y_proba_h2h))
print(classification_report(y_test_h2h, y_pred_h2h, target_names=['away_win','draw','home_win']))

(5900, 72) (1471, 72)
accuracy: 0.5098572399728076
log_loss: 1.008482813835144
              precision    recall  f1-score   support

    away_win       0.53      0.52      0.53       486
        draw       0.31      0.31      0.31       362
    home_win       0.61      0.62      0.61       623

    accuracy                           0.51      1471
   macro avg       0.48      0.48      0.48      1471
weighted avg       0.51      0.51      0.51      1471



In [48]:
importance_h2h = pd.Series(model_h2h.feature_importances_, index=feature_cols_h2h).sort_values(ascending=False)
print(importance_h2h.head(20))
print(importance_h2h[h2h_cols])

rank_gap                             0.153197
away_deep_expanding                  0.038625
home_xG_expanding                    0.036374
away_xG_expanding                    0.034736
home_deep_expanding                  0.034721
away_rank                            0.034357
away_ppda_allowed_value_expanding    0.029789
home_ppda_allowed_value_expanding    0.027545
h2h_xg_diff                          0.020554
away_xG_roll5                        0.020442
home_rank                            0.020258
away_deep_roll5                      0.019694
home_xGA_expanding                   0.019621
away_scored_expanding                0.019107
h2h_this_home_win_rate               0.018729
away_missed_expanding                0.018629
home_ppda_allowed_value_roll5        0.018587
home_xGA_roll5                       0.018395
home_missed_roll5                    0.018321
away_deep_allowed_expanding          0.017913
dtype: float32
h2h_matches               0.015366
h2h_this_home_win_rate    0.01

In [49]:
# already merged home_xG/away_xG onto matches earlier for H2H — bring in goals too
matches = matches.merge(
    matches_meta[['match_id', 'home_goals', 'away_goals']], on='match_id', how='left'
)

# Target 1: residual (goals - xG), from home team's perspective
matches['home_xg_residual'] = matches['home_goals'] - matches['home_xG']
matches['away_xg_residual'] = matches['away_goals'] - matches['away_xG']

# Target 2: clinical_rate direct (goals / xG) — guard against xG=0 edge case
matches['home_clinical_rate'] = matches['home_goals'] / matches['home_xG'].replace(0, np.nan)
matches['away_clinical_rate'] = matches['away_goals'] / matches['away_xG'].replace(0, np.nan)

# Target 3: tercile bucket, computed on home_xg_residual (consistent methodology with Notebook 06)
matches['home_performance_bucket'] = pd.qcut(
    matches['home_xg_residual'], q=3, labels=['under', 'neutral', 'over']
)

print(matches[['home_goals','home_xG','home_xg_residual','home_clinical_rate','home_performance_bucket']].describe(include='all'))
print(matches['home_clinical_rate'].isna().sum(), 'clinical_rate NaN (xG=0 matches)')

         home_goals      home_xG  home_xg_residual  home_clinical_rate  \
count   8982.000000  8982.000000       8982.000000         8980.000000   
unique          NaN          NaN               NaN                 NaN   
top             NaN          NaN               NaN                 NaN   
freq            NaN          NaN               NaN                 NaN   
mean       1.534625     1.601721         -0.067096            1.009269   
std        1.307722     0.946126          1.031733            1.012261   
min        0.000000     0.000000         -4.416440            0.000000   
25%        1.000000     0.889445         -0.733278            0.396126   
50%        1.000000     1.439810         -0.154351            0.897776   
75%        2.000000     2.148842          0.549555            1.404414   
max        9.000000     6.881890          4.781320           31.687987   

       home_performance_bucket  
count                     8982  
unique                       3  
top         

In [50]:
## Future Direction: AI Transfer Fit / Recommendation System
"""
Idea: rather than a generic "quality delta" for incoming/outgoing players, build a system that scores
player-team fit based on team-level departmental needs (attack/defense/passing) vs individual player
profiles in those same departments.

Scoping notes for when this is picked up:
- Need to confirm data availability for defensive on-ball stats per player (tackles, interceptions,
  duels) — players_enriched is currently attack/creation-leaning (npxG, xA, key_passes, xGChain).
  May need a new data pull from Understat or a supplementary source.
- "Team need" requires a defined baseline (league-average per position/department) to identify
  relative weakness, not just raw team totals.
- "Fit" scoring is a genuinely open problem — likely needs its own small model or scoring function,
  not a simple aggregation.
- Evaluation is non-trivial: no clean "did this transfer work" label exists; would need to define
  success metrics and a time window post-transfer.
- Natural fit for Phase 6 (agentic AI layer) — an LLM reasoning over team-need profiles qualitatively
  may complement or outperform a pure stats-based fit score, worth considering both.
- Strong MSP candidate: could stand alone as a demo ("given this team's current gaps, which available
  players best address them") distinct from the match-outcome/xG-performance modeling in Phases 4-5.
  """

'\nIdea: rather than a generic "quality delta" for incoming/outgoing players, build a system that scores\nplayer-team fit based on team-level departmental needs (attack/defense/passing) vs individual player\nprofiles in those same departments.\n\nScoping notes for when this is picked up:\n- Need to confirm data availability for defensive on-ball stats per player (tackles, interceptions,\n  duels) — players_enriched is currently attack/creation-leaning (npxG, xA, key_passes, xGChain).\n  May need a new data pull from Understat or a supplementary source.\n- "Team need" requires a defined baseline (league-average per position/department) to identify\n  relative weakness, not just raw team totals.\n- "Fit" scoring is a genuinely open problem — likely needs its own small model or scoring function,\n  not a simple aggregation.\n- Evaluation is non-trivial: no clean "did this transfer work" label exists; would need to define\n  success metrics and a time window post-transfer.\n- Natural fit f

In [51]:
home_perspective = matches.copy()
home_perspective['team_name'] = home_perspective['home_team']
home_perspective['opponent_name'] = home_perspective['away_team']
home_perspective['is_home'] = 1
home_perspective['goals'] = home_perspective['home_goals']
home_perspective['xG_actual'] = home_perspective['home_xG']
home_perspective['xg_residual'] = home_perspective['home_xg_residual']
home_perspective['clinical_rate'] = home_perspective['home_clinical_rate']

away_perspective = matches.copy()
away_perspective['team_name'] = away_perspective['away_team']
away_perspective['opponent_name'] = away_perspective['home_team']
away_perspective['is_home'] = 0
away_perspective['goals'] = away_perspective['away_goals']
away_perspective['xG_actual'] = away_perspective['away_xG']
away_perspective['xg_residual'] = away_perspective['away_xg_residual']
away_perspective['clinical_rate'] = away_perspective['away_clinical_rate']

def build_own_opp_cols(df, own_prefix, opp_prefix):
    rename_map = {}
    for m in roll_metrics:
        rename_map[f'{own_prefix}_{m}_roll5'] = f'own_{m}_roll5'
        rename_map[f'{own_prefix}_{m}_expanding'] = f'own_{m}_expanding'
        rename_map[f'{opp_prefix}_{m}_roll5'] = f'opp_{m}_roll5'
        rename_map[f'{opp_prefix}_{m}_expanding'] = f'opp_{m}_expanding'
    rename_map[f'{own_prefix}_rank'] = 'own_rank'
    rename_map[f'{opp_prefix}_rank'] = 'opp_rank'
    rename_map[f'{own_prefix}_rolling_pts_5'] = 'own_rolling_pts_5'
    rename_map[f'{opp_prefix}_rolling_pts_5'] = 'opp_rolling_pts_5'
    rename_map[f'{own_prefix}_form_xGD_divergence'] = 'own_form_xGD_divergence'
    rename_map[f'{opp_prefix}_form_xGD_divergence'] = 'opp_form_xGD_divergence'
    return df.rename(columns=rename_map)

home_perspective = build_own_opp_cols(home_perspective, 'home', 'away')
away_perspective = build_own_opp_cols(away_perspective, 'away', 'home')

model_b_cols = (
    ['match_id', 'date', 'league', 'year', 'team_name', 'opponent_name', 'is_home',
     'goals', 'xG_actual', 'xg_residual', 'clinical_rate', 'rank_gap'] +
    [f'own_{m}_roll5' for m in roll_metrics] + [f'own_{m}_expanding' for m in roll_metrics] +
    [f'opp_{m}_roll5' for m in roll_metrics] + [f'opp_{m}_expanding' for m in roll_metrics] +
    ['own_rank', 'opp_rank', 'own_rolling_pts_5', 'opp_rolling_pts_5',
     'own_form_xGD_divergence', 'opp_form_xGD_divergence'] +
    h2h_cols
)

model_b_data = pd.concat([home_perspective[model_b_cols], away_perspective[model_b_cols]], ignore_index=True)

# tercile bucket computed AFTER combining both perspectives, so it's consistent across home/away
model_b_data['performance_bucket'] = pd.qcut(model_b_data['xg_residual'], q=3, labels=['under', 'neutral', 'over'])

print(model_b_data.shape)
print(model_b_data[['xg_residual', 'clinical_rate']].describe())
print(model_b_data['performance_bucket'].value_counts())

(17964, 56)
        xg_residual  clinical_rate
count  17964.000000   17957.000000
mean      -0.044218       1.030745
std        0.976971       1.200137
min       -4.497340       0.000000
25%       -0.666400       0.000000
50%       -0.132431       0.898808
75%        0.540717       1.431608
max        4.934900      47.644004
performance_bucket
under      5988
neutral    5988
over       5988
Name: count, dtype: int64


In [52]:
print(model_b_data.nlargest(5, 'clinical_rate')[['team_name', 'opponent_name', 'goals', 'xG_actual', 'clinical_rate']])

              team_name    opponent_name  goals  xG_actual  clinical_rate
16815  Newcastle United   Crystal Palace      1   0.020989      47.644004
14218             Cadiz  Atletico Madrid      1   0.026046      38.394054
1538             Spezia            Inter      1   0.031558      31.687987
12592         Leicester          Chelsea      1   0.041797      23.925048
9328          Marseille       Strasbourg      1   0.042012      23.802496


In [53]:
print(model_b_data['year'].value_counts().sort_index())

year
2020    3652
2021    3652
2022    3652
2023    3504
2024    3504
Name: count, dtype: int64


In [54]:
test_year = 2024
train_b = model_b_data[model_b_data['year'] < test_year].copy()
test_b = model_b_data[model_b_data['year'] == test_year].copy()

print(train_b.shape, test_b.shape)
print(train_b['year'].unique(), test_b['year'].unique())

(14460, 56) (3504, 56)
[2020 2021 2022 2023] [2024]


In [55]:
feature_cols_b = (
    [f'own_{m}_roll5' for m in roll_metrics] + [f'own_{m}_expanding' for m in roll_metrics] +
    [f'opp_{m}_roll5' for m in roll_metrics] + [f'opp_{m}_expanding' for m in roll_metrics] +
    ['own_rank', 'opp_rank', 'rank_gap', 'own_rolling_pts_5', 'opp_rolling_pts_5',
     'own_form_xGD_divergence', 'opp_form_xGD_divergence'] + h2h_cols
)

train_b_clean = train_b.dropna(subset=feature_cols_b + ['xg_residual', 'clinical_rate', 'performance_bucket'])
test_b_clean = test_b.dropna(subset=feature_cols_b + ['xg_residual', 'clinical_rate', 'performance_bucket'])

print(train_b_clean.shape, test_b_clean.shape)

(11903, 56) (2833, 56)


In [56]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X_train_b = train_b_clean[feature_cols_b]
y_train_res = train_b_clean['xg_residual']
X_test_b = test_b_clean[feature_cols_b]
y_test_res = test_b_clean['xg_residual']

model_residual = XGBRegressor(
    n_estimators=200, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
model_residual.fit(X_train_b, y_train_res)
pred_res = model_residual.predict(X_test_b)

print('MAE:', mean_absolute_error(y_test_res, pred_res))
print('R2:', r2_score(y_test_res, pred_res))

# baseline: always predict the training mean (roughly zero, since goals ~ xG on average)
baseline_pred = np.full_like(y_test_res, y_train_res.mean())
print('baseline MAE:', mean_absolute_error(y_test_res, baseline_pred))
print('baseline R2:', r2_score(y_test_res, baseline_pred))

MAE: 0.7587065817093949
R2: -0.004809094478588127
baseline MAE: 0.7611933057885307
baseline R2: -0.00482899917269064


In [57]:
y_train_rate = train_b_clean['clinical_rate']
y_test_rate = test_b_clean['clinical_rate']

model_rate = XGBRegressor(
    n_estimators=200, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
model_rate.fit(X_train_b, y_train_rate)
pred_rate = model_rate.predict(X_test_b)

print('MAE:', mean_absolute_error(y_test_rate, pred_rate))
print('R2:', r2_score(y_test_rate, pred_rate))

baseline_pred_rate = np.full_like(y_test_rate, y_train_rate.mean())
print('baseline MAE:', mean_absolute_error(y_test_rate, baseline_pred_rate))

MAE: 0.7049073227887729
R2: -0.011314771279333558
baseline MAE: 0.7003705735599491


In [58]:
bucket_map = {'under': 0, 'neutral': 1, 'over': 2}
y_train_bucket = train_b_clean['performance_bucket'].map(bucket_map)
y_test_bucket = test_b_clean['performance_bucket'].map(bucket_map)

model_bucket = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3,
    n_estimators=200, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='mlogloss'
)
model_bucket.fit(X_train_b, y_train_bucket)
pred_bucket = model_bucket.predict(X_test_b)
proba_bucket = model_bucket.predict_proba(X_test_b)

print('accuracy:', accuracy_score(y_test_bucket, pred_bucket))
print('log_loss:', log_loss(y_test_bucket, proba_bucket))
print(classification_report(y_test_bucket, pred_bucket, target_names=['under', 'neutral', 'over']))

accuracy: 0.35545358277444405
log_loss: 1.094815969467163
              precision    recall  f1-score   support

       under       0.36      0.30      0.33      1013
     neutral       0.37      0.39      0.38       925
        over       0.33      0.38      0.35       895

    accuracy                           0.36      2833
   macro avg       0.36      0.36      0.36      2833
weighted avg       0.36      0.36      0.35      2833

